In [ ]:
import csv, math

class Node:
    def __init__(self, attr=None, ans=None):
        self.attr = attr
        self.ans = ans
        self.children = {}

def load(f):
    d = list(csv.reader(open(f)))
    return d[1:], d[0]

def entropy(data):
    labels = [r[-1] for r in data]
    n = len(labels)
    return -sum(labels.count(v)/n * math.log2(labels.count(v)/n) for v in set(labels))

def best_split(data, features):
    def gain(col):
        total = len(data)
        weighted = sum(len(sub := [r for r in data if r[col]==v]) / total * entropy(sub)
                       for v in set(r[col] for r in data))
        return entropy(data) - weighted
    return max(range(len(features)-1), key=gain)

def build(data, features):
    labels = [r[-1] for r in data]
    if len(set(labels)) == 1:
        return Node(ans=labels[0])
    if len(features) == 1:
        return Node(ans=max(set(labels), key=labels.count))
    b = best_split(data, features)
    node = Node(attr=features[b])
    for v in set(r[b] for r in data):
        subset = [r[:b] + r[b+1:] for r in data if r[b] == v]
        node.children[v] = build(subset, features[:b] + features[b+1:])
    return node

def print_tree(node, level=0):
    pad = "  " * level
    if node.ans:
        print(pad + "Answer:", node.ans)
        return
    print(pad + node.attr)
    for v, child in node.children.items():
        print(pad + "->", v)
        print_tree(child, level+1)

def classify(node, test, features):
    if node.ans:
        return node.ans
    i = features.index(node.attr)
    v = test[i]
    if v in node.children:
        return classify(node.children[v], test[:i]+test[i+1:], features[:i]+features[i+1:])
    return "Unknown"

# Main
data, features = load("data3.csv")
tree = build(data, features)

print("Decision Tree using ID3 Algorithm:")
print_tree(tree)

for test in load("data3_test.csv")[0]:
    print("\nTest Instance:", test)
    print("Predicted Label:", classify(tree, test, features))